In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import ttk, messagebox
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk

# ==============================================================================
# I. CosineThetaMagnet Class (Core Physics and Geometry)
# ==============================================================================

class CosineThetaMagnet:
    """
    Calculates the conductor coordinates, magnetic field, and harmonics 
    for a cosine-theta magnet based on user inputs.
    """
    
    def __init__(self, inputs):
        # Store all input parameters
        for key, value in inputs.items():
            setattr(self, key, value)
            
        # Physical constant: Permeability of free space
        self.mu0 = 4 * np.pi * 1e-7 # T*m/A
        
        # Will store the coordinates and current direction of ALL conductors
        self.conductor_coords = []
        
        # Calculate the geometry immediately upon creation
        self._calculate_geometry()
        
    
    def _calculate_geometry(self):
        """
        Calculates the (x, y) coordinates for all individual conductors.
        Conductors are stacked azimuthally, incorporating spacing (pitch).
        """
        
        self.conductor_coords = []
        n = self.n_pole
        
        # --- Conductor and Block Dimensions ---
        conductor_diameter = 2 * self.R_conductor
        conductor_pitch = self.conductor_pitch # 2*R_cond + S_cond
        
        # Radial thickness of one block layer is simply conductor_pitch
        radial_block_thickness = conductor_pitch
        
        # Radius of the center of each layer
        layer_radii = [self.R_inner + (k + 0.5) * radial_block_thickness
                       for k in range(self.n_layers)]
        
        # Angular parameters
        sector_angle = np.pi / n
        angle_step = sector_angle / self.n_blocks_per_layer
        
        
        for k, R_k in enumerate(layer_radii): # Iterate over layers (radial stacks)
            
            for j in range(self.n_blocks_per_layer): # Iterate over azimuthal blocks
                
                theta_center = angle_step * (j + 0.5)
                I_dir = 1.0 # Current out of the page (positive Z)

                
                # --- Conductor Placement within the Block (Azimuthal Stacking) ---
                
                # Total angle covered by the conductors in the block
                azimuthal_span_angle = (self.n_conductors_per_block) * conductor_pitch / R_k
                
                # The start angle of the first conductor's *center* in the block
                # Offset by half the total span angle to center the block on theta_center
                theta_start_center = theta_center - azimuthal_span_angle / 2 + (0.5 * conductor_pitch / R_k)
                
                for c in range(self.n_conductors_per_block):
                    
                    # 1. Calculate the conductor angle (center angle + offset by c * pitch)
                    theta_c = theta_start_center + c * (conductor_pitch / R_k)
                    
                    # 2. Radial position remains R_k (the layer center)
                    R_c = R_k 
                    
                    # Convert polar to Cartesian coordinates
                    x = R_c * np.cos(theta_c)
                    y = R_c * np.sin(theta_c)
                    
                    # Append (x, y, current_direction)
                    self.conductor_coords.append((x, y, I_dir * self.I_nominal))
                    
                    # --- Mirroring to create the full magnet (4-quadrant symmetry) ---
                    self.conductor_coords.append((-x, y, -I_dir * self.I_nominal))
                    self.conductor_coords.append((-x, -y, I_dir * self.I_nominal))
                    self.conductor_coords.append((x, -y, -I_dir * self.I_nominal))

                    
    def calculate_field_at_point(self, xp, yp):
        """
        Calculates the total magnetic field (Bx, By) at a single point (xp, yp) 
        due to all conductors using the Biot-Savart Law for infinite wires.
        """
        
        Bx_total = 0.0
        By_total = 0.0
        
        for xw, yw, Iw in self.conductor_coords:
            dx = xp - xw
            dy = yp - yw
            r_sq = dx**2 + dy**2
            
            if r_sq < 1e-18:
                continue 
            
            K = self.mu0 * Iw / (2 * np.pi)
            
            Bx_total += -K * dy / r_sq
            By_total += K * dx / r_sq
            
        return Bx_total, By_total

    def calculate_field_on_grid(self, x_grid, y_grid):
        """
        Calculates the magnetic field over a 2D grid of points.
        """
        Bx_map = np.zeros(x_grid.shape)
        By_map = np.zeros(y_grid.shape)
        
        for i in range(x_grid.shape[0]):
            for j in range(x_grid.shape[1]):
                xp, yp = x_grid[i, j], y_grid[i, j]
                
                if np.sqrt(xp**2 + yp**2) < self.R_inner:
                    Bx, By = self.calculate_field_at_point(xp, yp)
                    Bx_map[i, j] = Bx
                    By_map[i, j] = By
                    
        return Bx_map, By_map

    def calculate_harmonics(self, N_max=15, N_pts=360):
        """
        Calculates the normal (bn) and skew (an) field harmonics up to N_max.
        """
        
        R_ref = self.R_ref
        thetas = np.linspace(0, 2 * np.pi, N_pts, endpoint=False)
        x_pts = R_ref * np.cos(thetas)
        y_pts = R_ref * np.sin(thetas)
        complex_field = np.zeros(N_pts, dtype=complex)
        
        for j in range(N_pts):
            Bx, By = self.calculate_field_at_point(x_pts[j], y_pts[j])
            complex_field[j] = By + 1j * Bx

        coefficients = {}
        for n in range(1, N_max + 1):
            k = n - 1 
            C_n = np.sum(complex_field * np.exp(-1j * k * thetas)) / N_pts
            coefficients[n] = C_n
            
        n_main = self.n_pole
        if n_main not in coefficients: n_main = 1
        
        C_main = coefficients[n_main]
        B_main = C_main.real 
        
        if np.abs(B_main) < 1e-12:
            return {'normal_harmonics': {n: 0.0 for n in range(1, N_max + 1)}, 
                    'skew_harmonics': {n: 0.0 for n in range(1, N_max + 1)}}

        normal_harmonics = {}
        skew_harmonics = {}
        
        for n, C_n in coefficients.items():
            B_n = C_n.real
            A_n = C_n.imag
            
            b_n = (B_n / B_main) * 1e4
            a_n = (A_n / B_main) * 1e4
            
            normal_harmonics[n] = b_n
            skew_harmonics[n] = a_n
            
        return {'normal_harmonics': normal_harmonics, 'skew_harmonics': skew_harmonics}

    # --- Plotting Methods ---

    def plot_cross_section(self):
        """Plots the full 2D cross-section of the magnet conductors (circles)."""
        plt.figure()
        ax = plt.gca()
        
        R_inner = self.R_inner
        R_cond = self.R_conductor
        
        circle_inner = plt.Circle((0, 0), R_inner, color='k', fill=False, linestyle='--', linewidth=2, label='Inner Radius')
        ax.add_patch(circle_inner)

        legend_handles = []
        has_pos_I = False
        has_neg_I = False

        for xw, yw, Iw in self.conductor_coords:
            if Iw > 0:
                face_color = 'red'
                has_pos_I = True
            else:
                face_color = 'blue'
                has_neg_I = True
            
            # Use facecolor to avoid UserWarning while setting edgecolor
            conductor_circle = plt.Circle((xw, yw), R_cond, 
                                          facecolor=face_color, 
                                          edgecolor='k', 
                                          linewidth=0.5, 
                                          zorder=2) 
            ax.add_patch(conductor_circle)

        if has_pos_I:
            legend_out = plt.Circle((0, 0), 1, facecolor='red', edgecolor='k', label='+I (Out)')
            legend_handles.append(legend_out)
        
        if has_neg_I:
            legend_in = plt.Circle((0, 0), 1, facecolor='blue', edgecolor='k', label='-I (In)')
            legend_handles.append(legend_in)
            
        legend_handles.append(plt.Line2D([0], [0], color='k', linestyle='--', linewidth=2, label='Inner Radius'))

        coords = np.array([[x, y] for x, y, I in self.conductor_coords])
        max_coord = np.max(np.abs(coords)) if coords.size > 0 else R_inner 
        margin = R_cond * 2 * self.n_conductors_per_block * self.n_layers * 1.5 
        plot_limit = max(max_coord + R_cond, R_inner) + margin

        ax.set_xlim(-plot_limit, plot_limit)
        ax.set_ylim(-plot_limit, plot_limit)
        ax.set_aspect('equal', adjustable='box')
        
        plt.xlabel('X (m)')
        plt.ylabel('Y (m)')
        plt.title(f'{self.magnet_type.capitalize()} Coil Cross-Section (Layered $\\cos({self.n_pole}\\theta)$)', fontsize=8)
        plt.legend(handles=legend_handles, loc='upper right')
        plt.grid(True, linestyle=':', alpha=0.6)


    def plot_single_quadrant(self):
        """Plots the 2D cross-section of only the conductors in the first quadrant."""
        plt.figure()
        ax = plt.gca()
        
        R_inner = self.R_inner
        R_cond = self.R_conductor
        
        circle_inner = plt.Circle((0, 0), R_inner, color='k', fill=False, linestyle='--', linewidth=2, label='Inner Radius')
        ax.add_patch(circle_inner)

        quadrant_coords = [(x, y, I) for x, y, I in self.conductor_coords if x >= -1e-9 and y >= -1e-9]
        
        legend_handles = []
        has_pos_I = False
        has_neg_I = False

        for xw, yw, Iw in quadrant_coords:
            if Iw > 0:
                face_color = 'red'
                has_pos_I = True
            else:
                face_color = 'blue'
                has_neg_I = True
            
            conductor_circle = plt.Circle((xw, yw), R_cond, 
                                          facecolor=face_color, 
                                          edgecolor='k', 
                                          linewidth=0.5, 
                                          zorder=2)
            ax.add_patch(conductor_circle)

        if has_pos_I:
            legend_out = plt.Circle((0, 0), 1, facecolor='red', edgecolor='k', label='+I (Out)')
            legend_handles.append(legend_out)
        
        if has_neg_I:
            legend_in = plt.Circle((0, 0), 1, facecolor='blue', edgecolor='k', label='-I (In)')
            legend_handles.append(legend_in)
            
        legend_handles.append(plt.Line2D([0], [0], color='k', linestyle='--', linewidth=2, label='Inner Radius'))

        if quadrant_coords:
            coords = np.array([[x, y] for x, y, I in quadrant_coords])
            max_coord = np.max(coords)
            R_outer_max = max_coord + R_cond 
        else:
            R_outer_max = R_inner
            
        margin_factor = 1.15
        plot_limit = R_outer_max * margin_factor

        ax.set_xlim(-0.01, plot_limit) 
        ax.set_ylim(-0.01, plot_limit)
        ax.set_aspect('equal', adjustable='box')
        
        plt.xlabel('X (m)')
        plt.ylabel('Y (m)')
        plt.title(f'{self.magnet_type.capitalize()} Coil - First Quadrant View', fontsize=8)
        plt.legend(handles=legend_handles, loc='upper right')
        plt.grid(True, linestyle=':', alpha=0.6)


    def plot_field(self, Bx_map, By_map, x_grid, y_grid, n_quiver=10):
        """Plots the magnetic field: surface plot for magnitude and arrows for direction."""
        plt.figure()
        B_mag = np.sqrt(Bx_map**2 + By_map**2)
        
        plt.contourf(x_grid, y_grid, B_mag, levels=50, cmap='viridis')
        plt.colorbar(label='|B| (T)')
        
        circle_inner = plt.Circle((0, 0), self.R_inner, color='w', fill=False, linestyle='--', linewidth=1)
        plt.gca().add_patch(circle_inner)

        skip = int(x_grid.shape[0] / n_quiver)
        
        plt.quiver(x_grid[::skip, ::skip], y_grid[::skip, ::skip], 
                   Bx_map[::skip, ::skip], By_map[::skip, ::skip], 
                   color='r', scale=np.nanmax(B_mag)*2, alpha=0.7, headwidth=5)
        
        
        plt.xlabel('X (m)')
        plt.ylabel('Y (m)')
        plt.title(f'Magnetic Field Map ($R_{{ref}}={self.R_ref:.2f}$ m)', fontsize=8)
        plt.gca().set_aspect('equal', adjustable='box')


# ==============================================================================
# II. MagnetGUI Class (Tkinter Frontend)
# ==============================================================================

class MagnetGUI(tk.Tk):
    """
    Tkinter GUI for collecting magnet design inputs and triggering 
    calculation and plotting, displaying all results in one window.
    """
    def __init__(self, magnet_class):
        # Initialize the parent class (tk.Tk) explicitly
        tk.Tk.__init__(self) 
        
        self.title("🧲 Cosine-Theta Magnet Designer")
        self.magnet_class = magnet_class
        
        self.entries = {}
        
        self.params = [
            ("Inner Radius (R_inner) [m]:", "R_inner", 0.05),
            ("Magnet Type (Dipole=1, Quad=2, etc.):", "n_pole", 1),
            ("Number of Layers:", "n_layers", 2),
            ("Blocks per Layer:", "n_blocks_per_layer", 4),
            ("Conductors per Block:", "n_conductors_per_block", 10),
            ("Conductor Radius (R_cond) [m]:", "R_conductor", 0.001),
            ("Conductor Spacing (S_cond) [m]:", "S_conductor", 0.0001),
            ("Reference Radius (R_ref) [m]:", "R_ref", 0.03),
            ("Nominal Current (I_nominal) [A]:", "I_nominal", 5000)
        ]

        self._create_widgets()

    def _create_widgets(self):
        main_frame = ttk.Frame(self, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        self.columnconfigure(0, weight=1)
        self.rowconfigure(0, weight=1)
        
        # Input Fields
        for i, (label_text, key, default_value) in enumerate(self.params):
            label = ttk.Label(main_frame, text=label_text)
            label.grid(row=i, column=0, sticky=tk.W, pady=2, padx=5)
            
            entry = ttk.Entry(main_frame, width=20)
            entry.insert(0, str(default_value))
            entry.grid(row=i, column=1, sticky=(tk.W, tk.E), pady=2, padx=5)
            self.entries[key] = entry
            
        # Magnet Type Dropdown 
        ttk.Label(main_frame, text="Pole Type:").grid(row=len(self.params), column=0, sticky=tk.W, pady=5, padx=5)
        self.magnet_type_var = tk.StringVar(self)
        self.magnet_type_var.set("dipole") 
        
        pole_types = {"dipole": 1, "quadrupole": 2, "sextupole": 3, "octupole": 4}
        type_options = list(pole_types.keys())
        
        type_dropdown = ttk.Combobox(main_frame, textvariable=self.magnet_type_var, values=type_options, width=18)
        type_dropdown.grid(row=len(self.params), column=1, sticky=(tk.W, tk.E), pady=5, padx=5)
        
        def update_n_pole(*args):
            selected_type = self.magnet_type_var.get()
            n_pole = pole_types.get(selected_type, 1)
            self.entries["n_pole"].delete(0, tk.END)
            self.entries["n_pole"].insert(0, str(n_pole))

        self.magnet_type_var.trace_add("write", update_n_pole)
            
        # Calculate Button
        calc_button = ttk.Button(main_frame, text="Calculate & Plot Magnet", command=self._run_calculation)
        calc_button.grid(row=len(self.params) + 1, column=0, columnspan=2, pady=10)


    def _validate_and_get_inputs(self):
        """Validates all numerical inputs and returns a dictionary."""
        inputs = {}
        try:
            for key, entry in self.entries.items():
                if key in ["n_pole", "n_layers", "n_blocks_per_layer", "n_conductors_per_block"]:
                    inputs[key] = int(entry.get())
                else:
                    inputs[key] = float(entry.get())
            
            inputs['magnet_type'] = self.magnet_type_var.get()
            inputs['conductor_pitch'] = 2 * inputs['R_conductor'] + inputs['S_conductor']

            # Basic Validation
            if inputs['R_inner'] <= 0 or inputs['R_conductor'] <= 0 or inputs['R_ref'] <= 0:
                raise ValueError("All radii must be positive.")
            if inputs['R_ref'] >= inputs['R_inner']:
                raise ValueError("Reference Radius (R_ref) must be strictly less than Inner Radius (R_inner).")
            if inputs['n_pole'] <= 0:
                raise ValueError("Pole number must be a positive integer.")
                
            return inputs
            
        except ValueError as e:
            messagebox.showerror("Input Error", f"Invalid input value or constraint violation:\n{e}")
            return None


    def _run_calculation(self):
        """
        Gathers inputs, runs the magnet calculation, and prepares all results 
        to be displayed in a single results window.
        """
        inputs = self._validate_and_get_inputs()
        if not inputs:
            return

        try:
            print("\n--- Starting Magnet Calculation ---")
            
            # 1. Initialize Magnet and Calculate Geometry
            magnet = self.magnet_class(inputs)
            print(f"Total individual conductors calculated: {len(magnet.conductor_coords)}")

            # 2. Setup Grid for Field Calculation
            R_grid_max = magnet.R_inner * 1.0
            N_points = 50 
            x = np.linspace(-R_grid_max, R_grid_max, N_points)
            y = np.linspace(-R_grid_max, R_grid_max, N_points)
            X, Y = np.meshgrid(x, y)
            
            print("Calculating Magnetic Field on Grid...")
            Bx_map, By_map = magnet.calculate_field_on_grid(X, Y)

            # 3. Calculate Harmonics
            print("Running Harmonic Analysis...")
            harmonics = magnet.calculate_harmonics(N_max=15)
            
            # 4. Generate Plots (Figures must be created before displaying them)
            # A. Full Coil Cross-Section
            plt.figure(1, figsize=(4.5, 4.5))
            magnet.plot_cross_section()
            fig1 = plt.gcf()
            
            # B. Single Quadrant View
            plt.figure(2, figsize=(4.5, 4.5))
            magnet.plot_single_quadrant()
            fig2 = plt.gcf()

            # C. Field Map
            plt.figure(3, figsize=(4.5, 4.5))
            magnet.plot_field(Bx_map, By_map, X, Y, n_quiver=10)
            fig3 = plt.gcf()
            
            # Close the figures so they don't appear in separate windows later
            plt.close(fig1)
            plt.close(fig2)
            plt.close(fig3)
            
            figures = [fig1, fig2, fig3]

            # 5. Display All Results in one popup window
            self._display_results(figures, harmonics, magnet.n_pole)
            
            print("--- Calculation Complete. Results Window Displayed. ---")

        except Exception as e:
            messagebox.showerror("Calculation Error", f"An error occurred during calculation:\n{e}")

    def _display_results(self, figures, harmonics, n_main):
        """Displays all plots and the harmonic table in a single Tkinter Toplevel window."""
        results_window = tk.Toplevel(self)
        results_window.title("Magnet Simulation Results")
        results_window.geometry("1400x800") 
        
        main_frame = ttk.Frame(results_window, padding="10")
        main_frame.pack(fill='both', expand=True)

        # --- Top Section: Plots (3 Figures) ---
        plot_frame = ttk.Frame(main_frame)
        plot_frame.pack(fill='x', pady=5)
        
        for i, fig in enumerate(figures):
            # Embed the figure into the plot frame
            plot_container = ttk.Frame(plot_frame)
            plot_container.grid(row=0, column=i, padx=5, sticky="nsew")
            
            # Ensure the figure titles are set for clarity in the embedded plot
            if i == 0:
                fig.suptitle("Coil Cross-Section", fontsize=10)
            elif i == 1:
                fig.suptitle("Single Quadrant View", fontsize=10)
            elif i == 2:
                fig.suptitle("Magnetic Field Map", fontsize=10)
            
            canvas = FigureCanvasTkAgg(fig, master=plot_container)
            canvas_widget = canvas.get_tk_widget()
            canvas_widget.pack(side=tk.TOP, fill=tk.BOTH, expand=True)
            
            # Add a Matplotlib toolbar for zooming/panning
            toolbar = NavigationToolbar2Tk(canvas, plot_container)
            toolbar.update()
            canvas_widget.pack(side=tk.TOP, fill=tk.BOTH, expand=True)
        
        # Configure the plot_frame columns to expand equally
        plot_frame.grid_columnconfigure(0, weight=1)
        plot_frame.grid_columnconfigure(1, weight=1)
        plot_frame.grid_columnconfigure(2, weight=1)

        # --- Bottom Section: Harmonic Table ---
        harm_frame = ttk.Frame(main_frame, padding="10")
        harm_frame.pack(fill='both', expand=True, pady=10)
        
        main_field_norm = harmonics['normal_harmonics'][n_main] / 1e4
        label_text = f"Main Harmonic (n={n_main}) Field Component (B_{n_main}): {main_field_norm:.4e} T"
        ttk.Label(harm_frame, text=label_text, font=('Arial', 10, 'bold')).pack(pady=5)

        # Create Treeview widget for the table
        tree = ttk.Treeview(harm_frame, columns=('n', 'bn', 'an'), show='headings')
        tree.heading('n', text='Harmonic (n)')
        tree.heading('bn', text='Normal (bn) [units]')
        tree.heading('an', text='Skew (an) [units]')
        tree.column('n', width=100, anchor='center')
        tree.column('bn', width=150, anchor='center')
        tree.column('an', width=150, anchor='center')
        
        for n in range(1, 16):
            bn_val = harmonics['normal_harmonics'].get(n, 0.0)
            an_val = harmonics['skew_harmonics'].get(n, 0.0)
            tree.insert('', tk.END, values=(n, f"{bn_val:.4f}", f"{an_val:.4f}"))

        tree.pack(padx=10, pady=10, fill='x', expand=False)


# ==============================================================================
# III. Main Execution Block
# ==============================================================================

if __name__ == "__main__":
    app = MagnetGUI(CosineThetaMagnet)
    app.mainloop()


--- Starting Magnet Calculation ---
Total individual conductors calculated: 320
Calculating Magnetic Field on Grid...
Running Harmonic Analysis...
--- Calculation Complete. Results Window Displayed. ---


<Figure size 450x450 with 0 Axes>